In [1]:
! pip install torch torchvision

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import numpy as np
from tqdm import tqdm

In [6]:
  class SingleRNN(nn.Module):
    def __init__(self, n_inputs, n_neurons):
      super(SingleRNN, self).__init__()
      self.Wx = torch.randn(n_inputs, n_neurons)
      self.Wy = torch.randn(n_neurons, n_neurons)
      self.b = torch.zeros(1, n_neurons)


    def forward(self, X0, X1):
      self.Y0 = torch.tanh(torch.mm(X0, self.Wx) + self.b)
      self.Y1 = torch.tanh(torch.mm(self.Y0, self.Wy  ) +
                          torch.mm(X1, self.Wx) + self.b )

      return self.Y0, self.Y1

In [7]:
N_INPUT = 4
N_NEURONS = 1

X0_batch = torch.tensor([[0,1,2,0],[3,4,5,0],[6,7,8,0],[9,0,1,0]],
                        dtype = torch.float)
X1_batch = torch.tensor([[9,8,7,0],[0,0,0,0],[6,5,4,0],[3,2,1,0]],
                         dtype = torch.float)

model = SingleRNN(N_INPUT, N_NEURONS)
Y0_val, Y1_val = model(X0_batch, X1_batch)

In [8]:
print(Y0_val)
print(Y1_val)


tensor([[ 0.9971],
        [ 0.9637],
        [ 0.6142],
        [-1.0000]])
tensor([[-1.0000],
        [-0.4061],
        [-1.0000],
        [-0.9995]])


## `#BASIC RNN`

In [10]:
class BasicRNN(nn.Module):
  def __init__(self,n_inputs,n_neurons):
    super(BasicRNN,self).__init__()
    self.Wx=torch.randn(n_inputs,n_neurons)
    self.Wy=torch.randn(n_neurons,n_neurons)
    self.b=torch.zeros(1,n_neurons)
  def forward(self,X0,X1):
    self.Y0=torch.tanh(torch.mm(X0,self.Wx)+self.b)
    self.Y1=torch.tanh(torch.mm(self.Y0,self.Wy)+torch.mm(X1,self.Wx)+self.b)
    return self.Y0, self.Y1

In [14]:
N_INPUT = 3
N_NEURONS = 5

X0_batch = torch.tensor([[0,1,2], [3,4,5],
                         [6,7,8], [9,0,1]],
                        dtype = torch.float)

X1_batch = torch.tensor([[9,8,7], [0,0,0],
                         [6,5,4], [3,2,1]],
                        dtype = torch.float)

model = BasicRNN(N_INPUT, N_NEURONS)

Y0_val, Y1_val = model(X0_batch, X1_batch)

In [15]:
print(Y0_val)
print(Y1_val)

tensor([[ 0.9988,  0.9122, -0.9565, -0.9798, -0.9764],
        [ 1.0000,  1.0000, -1.0000, -1.0000, -1.0000],
        [ 1.0000,  1.0000, -1.0000, -1.0000, -1.0000],
        [-0.9997,  1.0000, -1.0000, -1.0000, -0.9999]])
tensor([[ 1.0000,  1.0000, -1.0000, -1.0000, -1.0000],
        [-0.1404,  0.9641,  0.8005, -0.6987,  0.9942],
        [ 0.9999,  1.0000, -1.0000, -1.0000, -1.0000],
        [-0.9631,  1.0000, -0.9999, -0.9972, -1.0000]])


## PyTorch Built in RNN Cell

In [3]:
import torch
import torch.nn as nn

rnn = nn.RNNCell(3,5)

X_batch = torch.tensor([
    [[0,1,2], [3,4,5],
     [6,7,8], [9,0,1]],

    [[9,8,7], [0,0,0],
     [6,5,4],[3,2,1]]
], dtype = torch.float)

hx = torch.randn(4,5)
output = []
for i in range(2):
  hx = rnn(X_batch[i],hx)
  output.append(hx)

print(output)

[tensor([[ 0.8853, -0.2270,  0.8378,  0.5327,  0.8983],
        [ 1.0000, -0.8932,  0.9939,  0.9999,  0.6962],
        [ 1.0000, -0.9991,  0.9982,  1.0000,  0.9163],
        [ 0.9999, -0.9998, -0.5626,  0.9999, -0.9990]],
       grad_fn=<TanhBackward0>), tensor([[ 1.0000, -0.9999,  0.9982,  1.0000, -0.6406],
        [ 0.9163,  0.4200, -0.0190,  0.6020,  0.3986],
        [ 1.0000, -0.9931,  0.9512,  1.0000, -0.2906],
        [ 0.9927, -0.8258,  0.3912,  0.9975, -0.4654]],
       grad_fn=<TanhBackward0>)]


In [6]:
class CleanBasicRNN(nn.Module):
  def __init__(self, batch_size, n_inputs, n_neurons):
    super(CleanBasicRNN, self). __init__()
    self.rnn = nn.RNNCell(n_inputs, n_neurons)
    self.hx = torch.randn(batch_size, n_neurons)

  def forward(self,X):
    output=[]

    for  i in range(2):
      self.hx = self.rnn(X[i], self.hx)
      output.append(self.hx)

    return output, self.hx

In [7]:
FIXED_BATCH_SIZE = 4
N_INPUT = 3
N_NEURONS = 5

X_batch = torch.tensor([[[0,1,2], [3,4,5],
                         [6,7,8], [9,0,1]],
                        [[9,8,7], [0,0,0],
                         [6,5,4], [3,2,1]]
                       ], dtype = torch.float) # X0 and X1


model = CleanBasicRNN(FIXED_BATCH_SIZE, N_INPUT, N_NEURONS)
output_val, states_val = model(X_batch)
print(output_val)
print(states_val)

[tensor([[ 0.1494,  0.9442, -0.2750,  0.8976,  0.9115],
        [ 0.9848,  0.9980, -0.6455,  0.9872,  0.9954],
        [ 0.9937,  1.0000, -0.3989,  0.9999,  0.9999],
        [ 0.9946,  0.9974, -0.9852,  0.9866,  0.6955]],
       grad_fn=<TanhBackward0>), tensor([[ 0.9995,  1.0000, -0.5414,  1.0000,  0.9999],
        [-0.2567, -0.0529,  0.1579,  0.3004, -0.0528],
        [ 0.9901,  0.9996, -0.2976,  0.9975,  0.9925],
        [ 0.7824,  0.9567,  0.0736,  0.9178,  0.7737]],
       grad_fn=<TanhBackward0>)]
tensor([[ 0.9995,  1.0000, -0.5414,  1.0000,  0.9999],
        [-0.2567, -0.0529,  0.1579,  0.3004, -0.0528],
        [ 0.9901,  0.9996, -0.2976,  0.9975,  0.9925],
        [ 0.7824,  0.9567,  0.0736,  0.9178,  0.7737]],
       grad_fn=<TanhBackward0>)
